# Previsão de Diabetes Mellitus — CDC BRFSS 2015
## Rastreamento de Experimentos com MLflow

---

**Objetivo:** Registrar todas as decisões, métricas e artefatos do projeto em um servidor MLflow, criando um histórico completo e reproduzível de todos os experimentos realizados.

**O que será registrado:**
- Métricas do baseline (Regressão Logística)
- Métricas de todos os modelos comparados
- Trials do Optuna com a métrica composta
- Modelo final com hiperparâmetros e threshold
- Artefatos: pipeline, modelo, metadados

**Por que MLflow?**  
Sem rastreamento, experimentos de ML são difíceis de reproduzir e comparar. O MLflow resolve isso centralizando métricas, parâmetros e artefatos de cada execução em uma UI visual

## Instalação e Configuração

In [ ]:
!pip install mlflow pyngrok lightgbm scikit-learn joblib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json, subprocess, threading, time
import numpy  as np
import pandas as pd
import joblib
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from xgboost                 import XGBClassifier
from lightgbm                import LGBMClassifier
from sklearn.metrics         import (
    roc_auc_score, recall_score, precision_score,
    f1_score, accuracy_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.base            import BaseEstimator, TransformerMixin

SEED = 42
np.random.seed(SEED)
print('✅ Imports OK')

✅ Imports OK


## Iniciar Servidor MLflow com Túnel Público

No Colab não é possível acessar portas locais diretamente. Usei o **ngrok** para criar um túnel público que expõe a UI do MLflow no navegador.

**Passo necessário:** crie uma conta gratuita em [ngrok.com](https://ngrok.com) e copie seu authtoken em: Dashboard → Your Authtoken.

In [ ]:
# Configurar ngrok
# Cole seu authtoken gratuito do ngrok abaixo
NGROK_TOKEN = '3C3FaifZu3gN2IRnwCBcCHBwbTP_LHk9DvfVQNybV6pTuCkA'

!ngrok authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
os.makedirs('mlruns', exist_ok=True)
os.environ["MLFLOW_SERVER_ALLOWED_HOSTS"] = "*"

def subir_mlflow():
    subprocess.run(
        [
            'mlflow', 'ui',
            '--host', '0.0.0.0',
            '--port', '5000',
            '--allowed-hosts', '*'

        ]
    )

thread = threading.Thread(target=subir_mlflow, daemon=True)
thread.start()

time.sleep(5)

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print(public_url)

NgrokTunnel: "https://mercedez-unchagrined-indefectibly.ngrok-free.dev" -> "http://localhost:5000"


## Carregamento dos Artefatos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Redefine classe customizada antes de carregar o pipeline
class OutlierClipper(BaseEstimator, TransformerMixin):
    """Winsorização por percentis — mesma classe do pré-processamento."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.lower_ = X_df.quantile(self.lower)
        self.upper_ = X_df.quantile(self.upper)
        return self

    def transform(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        return X_df.clip(lower=self.lower_, upper=self.upper_, axis=1).values


# Carrega artefatos da etapa de modelagem
X_train      = np.load('/content/drive/MyDrive/artefatos/X_train.npy')
X_test       = np.load('/content/drive/MyDrive/artefatos/X_test.npy')
y_train      = np.load('/content/drive/MyDrive/artefatos/y_train.npy')
y_test       = np.load('/content/drive/MyDrive/artefatos/y_test.npy')
class_weight = joblib.load('/content/drive/MyDrive/artefatos/class_weight.pkl')
features     = joblib.load('/content/drive/MyDrive/artefatos/features.pkl')
pipeline     = joblib.load('/content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl')
modelo_final = joblib.load('/content/drive/MyDrive/artefatos/modelo_final.pkl')

with open('/content/drive/MyDrive/artefatos/metadados_modelo.json') as f:
    meta = json.load(f)

THRESH_OTIMO  = meta['threshold_otimo']
RECALL_MINIMO = meta['recall_teste']
scale_pos     = class_weight[1] / class_weight[0]

print(f'✅ Artefatos carregados')
print(f'   Modelo final  : {meta["modelo"]}')
print(f'   ROC-AUC       : {meta["roc_auc_teste"]}')
print(f'   Recall        : {meta["recall_teste"]}')
print(f'   Threshold     : {THRESH_OTIMO}')

✅ Artefatos carregados
   Modelo final  : LightGBM
   ROC-AUC       : 0.8192
   Recall        : 0.8168
   Threshold     : 0.473


## Experimento 1 — Baseline

Registra o baseline (Regressão Logística) como ponto de referência para todos os experimentos subsequentes.

In [ ]:
EXPERIMENTO = 'diabetes-brfss-2015'
mlflow.set_experiment(EXPERIMENTO)

# Treina e registra o baseline
baseline = LogisticRegression(
    C=1.0, max_iter=1000,
    class_weight=class_weight,
    random_state=SEED
)
baseline.fit(X_train, y_train)
y_prob_bl = baseline.predict_proba(X_test)[:, 1]
y_pred_bl = baseline.predict(X_test)

with mlflow.start_run(run_name='baseline_logistic_regression'):
    # Tags descritivas
    mlflow.set_tags({
        'tipo'      : 'baseline',
        'dataset'   : 'CDC BRFSS 2015',
        'etapa'     : 'comparacao_inicial',
        'autor'     : 'diabetes-project',
    })

    # Parâmetros
    mlflow.log_params({
        'modelo'        : 'LogisticRegression',
        'C'             : 1.0,
        'max_iter'      : 1000,
        'class_weight'  : 'balanced',
        'threshold'     : 0.5,
        'n_features'    : len(features),
        'n_treino'      : X_train.shape[0],
    })

    # Métricas
    mlflow.log_metrics({
        'roc_auc'  : roc_auc_score(y_test, y_prob_bl),
        'recall'   : recall_score(y_test, y_pred_bl),
        'precisao' : precision_score(y_test, y_pred_bl),
        'f1'       : f1_score(y_test, y_pred_bl),
        'acuracia' : accuracy_score(y_test, y_pred_bl),
    })

    # Modelo
    signature = infer_signature(X_train, baseline.predict_proba(X_train))
    mlflow.sklearn.log_model(baseline, 'modelo', signature=signature)

print('✅ Baseline registrado no MLflow')
print(f'   ROC-AUC : {roc_auc_score(y_test, y_prob_bl):.4f}')
print(f'   Recall  : {recall_score(y_test, y_pred_bl):.4f}')

2026/04/08 05:05:17 INFO mlflow.tracking.fluent: Experiment with name 'diabetes-brfss-2015' does not exist. Creating a new experiment.
2026/04/08 05:05:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:05:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline_logistic_regression at: http://localhost:5000/#/experiments/1/runs/610fa06af17e47399e2c8d532075a2d0
🧪 View experiment at: http://localhost:5000/#/experiments/1
✅ Baseline registrado no MLflow
   ROC-AUC : 0.8125
   Recall  : 0.7669


## Experimento 2 — Comparação de Modelos

Registra cada modelo como uma run separada dentro do mesmo experimento, permitindo comparação visual na UI do MLflow.

In [ ]:
MODELOS = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=50,
        class_weight=class_weight, random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_leaf=30, n_jobs=-1,
        class_weight=class_weight, random_state=SEED
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=5, subsample=0.8, random_state=SEED
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=SEED, verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, num_leaves=50,
        subsample=0.8, colsample_bytree=0.8,
        class_weight=class_weight,
        random_state=SEED, verbose=-1
    ),
}

recall_minimo = recall_score(y_test, y_pred_bl)  # recall do baseline

for nome, modelo in MODELOS.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)
    rec    = recall_score(y_test, y_pred)

    # Classifica se passou no critério clínico
    aprovado = rec >= recall_minimo

    with mlflow.start_run(run_name=f'comparacao_{nome.lower().replace(" ", "_")}'):
        mlflow.set_tags({
            'tipo'            : 'comparacao_inicial',
            'dataset'         : 'CDC BRFSS 2015',
            'aprovado_recall' : str(aprovado),
        })

        mlflow.log_params({
            'modelo'      : nome,
            'threshold'   : 0.5,
            'class_weight': 'balanced',
            'n_features'  : len(features),
        })

        mlflow.log_metrics({
            'roc_auc'            : auc,
            'recall'             : rec,
            'precisao'           : precision_score(y_test, y_pred),
            'f1'                 : f1_score(y_test, y_pred),
            'acuracia'           : accuracy_score(y_test, y_pred),
            'ganho_auc_baseline' : auc - roc_auc_score(y_test, y_prob_bl),
        })

        # Salva modelo apenas dos aprovados no critério de Recall
        if aprovado:
            signature = infer_signature(X_train, modelo.predict_proba(X_train))
            mlflow.sklearn.log_model(modelo, 'modelo', signature=signature)

    status = '✅' if aprovado else '❌'
    print(f'  {status} {nome:<22} | AUC: {auc:.4f} | Recall: {rec:.4f}')

print('\n✅ Comparação registrada no MLflow!')

2026/04/08 05:06:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:06:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run comparacao_decision_tree at: http://localhost:5000/#/experiments/1/runs/5c47b43d2303454294413c6bb7020013
🧪 View experiment at: http://localhost:5000/#/experiments/1
  ✅ Decision Tree          | AUC: 0.8080 | Recall: 0.7789


2026/04/08 05:07:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:07:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run comparacao_random_forest at: http://localhost:5000/#/experiments/1/runs/ccdc3a24c3fe416b95519cbe9211f8db
🧪 View experiment at: http://localhost:5000/#/experiments/1
  ✅ Random Forest          | AUC: 0.8163 | Recall: 0.7733
🏃 View run comparacao_gradient_boosting at: http://localhost:5000/#/experiments/1/runs/65db4a98d79e45ea86f9d641d2f2808d
🧪 View experiment at: http://localhost:5000/#/experiments/1
  ❌ Gradient Boosting      | AUC: 0.8195 | Recall: 0.1670


2026/04/08 05:08:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:08:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run comparacao_xgboost at: http://localhost:5000/#/experiments/1/runs/6c41a29e642b44b2aec9d296db1b02df
🧪 View experiment at: http://localhost:5000/#/experiments/1
  ✅ XGBoost                | AUC: 0.8188 | Recall: 0.7847


2026/04/08 05:09:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:09:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run comparacao_lightgbm at: http://localhost:5000/#/experiments/1/runs/e46b2bed3d2f4d05b49ee1d96b5a1a58
🧪 View experiment at: http://localhost:5000/#/experiments/1
  ✅ LightGBM               | AUC: 0.8190 | Recall: 0.7843

✅ Comparação registrada no MLflow!


## Experimento 3 — Trials do Optuna

Registra cada trial do Optuna como uma run separada, permitindo visualizar a evolução da busca de hiperparâmetros na UI do MLflow.

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.5 MB/s eta 0:00:00


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def build_lightgbm(trial):
    """Constrói LightGBM com hiperparâmetros sugeridos pelo Optuna."""
    return LGBMClassifier(
        n_estimators      = trial.suggest_int('n_estimators', 100, 500),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 8),
        num_leaves        = trial.suggest_int('num_leaves', 20, 100),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 10, 100),
        class_weight=class_weight, random_state=SEED, verbose=-1
    )


def objective_mlflow(trial):
    """
    Função objetivo com logging no MLflow.
    Cada trial = uma run no MLflow.
    """
    modelo = build_lightgbm(trial)
    cv_3   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    scores = cross_validate(
        modelo, X_train, y_train,
        cv=cv_3,
        scoring={'auc': 'roc_auc', 'recall': 'recall', 'f1': 'f1'},
        n_jobs=-1
    )

    auc_cv    = scores['test_auc'].mean()
    recall_cv = scores['test_recall'].mean()
    f1_cv     = scores['test_f1'].mean()

    # Descarta trial com Recall abaixo do baseline
    metrica = 0.0 if recall_cv < recall_minimo \
              else 0.5 * auc_cv + 0.3 * recall_cv + 0.2 * f1_cv

    # Registra trial no MLflow
    with mlflow.start_run(
        run_name=f'optuna_trial_{trial.number:03d}',
        nested=True
    ):
        mlflow.set_tags({
            'tipo'    : 'optuna_trial',
            'modelo'  : 'LightGBM',
            'valido'  : str(metrica > 0.0),
        })
        mlflow.log_params(trial.params)
        mlflow.log_metrics({
            'metrica_composta' : metrica,
            'auc_cv'           : auc_cv,
            'recall_cv'        : recall_cv,
            'f1_cv'            : f1_cv,
        })

    return metrica


print('🔍 Iniciando Optuna com logging no MLflow (80 trials)...')

with mlflow.start_run(run_name='optuna_lightgbm_tuning'):
    mlflow.set_tags({
        'tipo'   : 'optuna_study',
        'modelo' : 'LightGBM',
    })
    mlflow.log_params({
        'n_trials'        : 80,
        'metrica_objetivo': '0.5*AUC + 0.3*Recall + 0.2*F1',
        'recall_minimo'   : recall_minimo,
        'sampler'         : 'TPESampler',
    })

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective_mlflow, n_trials=80, show_progress_bar=True)

    trials_validos = [t for t in study.trials if t.value > 0.0]

    mlflow.log_metrics({
        'melhor_metrica_composta' : study.best_value,
        'trials_validos'          : len(trials_validos),
        'trials_descartados'      : 80 - len(trials_validos),
    })
    mlflow.log_params({f'best_{k}': v for k, v in study.best_params.items()})

print(f'\n✅ Optuna concluído!')
print(f'   Melhor métrica composta: {study.best_value:.4f}')
print(f'   Trials válidos: {len(trials_validos)}/80')

🔍 Iniciando Optuna com logging no MLflow (80 trials)...


  0%|          | 0/80 [00:00<?, ?it/s]

🏃 View run optuna_trial_000 at: http://localhost:5000/#/experiments/1/runs/765bff1090d0475286cbd4cbdcaa4eff
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run optuna_trial_001 at: http://localhost:5000/#/experiments/1/runs/f168eddb80a14b42ac61612954fd5795
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run optuna_trial_002 at: http://localhost:5000/#/experiments/1/runs/afc662986278434ca6bfadb056602d70
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run optuna_trial_003 at: http://localhost:5000/#/experiments/1/runs/c144a88abf6a4dc7bf58607c34c56234
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run optuna_trial_004 at: http://localhost:5000/#/experiments/1/runs/20fd29e6918145dc8d12b5bd897d5381
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run optuna_trial_005 at: http://localhost:5000/#/experiments/1/runs/d38b6497c21e406384f77ba85d800b3a
🧪 View experiment at: http://localhost:5000/#/experi

## Experimento 4 — Modelo Final

Registra o modelo final com os melhores hiperparâmetros, threshold otimizado e todas as métricas finais. Esta run é a versão oficial para produção.

In [ ]:
# Retreina com melhores hiperparâmetros
modelo_tunado = LGBMClassifier(
    **study.best_params,
    class_weight=class_weight,
    random_state=SEED, verbose=-1
)
modelo_tunado.fit(X_train, y_train)

y_prob_final = modelo_tunado.predict_proba(X_test)[:, 1]
y_pred_final = (y_prob_final >= THRESH_OTIMO).astype(int)

with mlflow.start_run(run_name='modelo_final_producao'):
    mlflow.set_tags({
        'tipo'      : 'modelo_final',
        'modelo'    : 'LightGBM',
        'status'    : 'producao',
        'dataset'   : 'CDC BRFSS 2015',
        'n_pacientes': '229474',
    })

    # Hiperparâmetros finais
    mlflow.log_params({
        **study.best_params,
        'threshold_otimo'  : THRESH_OTIMO,
        'class_weight'     : 'balanced',
        'metrica_threshold': 'max_recall_com_precisao_minima',
        'precisao_minima'  : round(y_test.mean() * 2, 4),
        'n_features'       : len(features),
        'n_treino'         : X_train.shape[0],
        'n_teste'          : X_test.shape[0],
    })

    # Métricas com threshold padrão
    y_pred_050 = (y_prob_final >= 0.5).astype(int)
    mlflow.log_metrics({
        'roc_auc'              : roc_auc_score(y_test, y_prob_final),
        'recall_thresh_050'    : recall_score(y_test, y_pred_050),
        'precisao_thresh_050'  : precision_score(y_test, y_pred_050),
        'f1_thresh_050'        : f1_score(y_test, y_pred_050),
        # Métricas com threshold ótimo
        'recall_thresh_otimo'  : recall_score(y_test, y_pred_final),
        'precisao_thresh_otimo': precision_score(y_test, y_pred_final),
        'f1_thresh_otimo'      : f1_score(y_test, y_pred_final),
        'acuracia_thresh_otimo': accuracy_score(y_test, y_pred_final),
        # Ganhos vs baseline
        'ganho_auc_vs_baseline'   : roc_auc_score(y_test, y_prob_final) - roc_auc_score(y_test, y_prob_bl),
        'ganho_recall_vs_baseline': recall_score(y_test, y_pred_final) - recall_score(y_test, y_pred_bl),
    })

    # Artefatos
    signature = infer_signature(X_train, y_prob_final)
    mlflow.lightgbm.log_model(modelo_tunado, 'modelo_lightgbm', signature=signature)
    mlflow.log_artifact('/content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl', 'artefatos')
    mlflow.log_artifact('/content/drive/MyDrive/artefatos/features.pkl',                  'artefatos')
    mlflow.log_artifact('/content/drive/MyDrive/artefatos/class_weight.pkl',              'artefatos')
    mlflow.log_artifact('/content/drive/MyDrive/artefatos/metadados_modelo.json',         'artefatos')

    run_id = mlflow.active_run().info.run_id

print('✅ Modelo final registrado no MLflow!')
print(f'   Run ID    : {run_id}')
print(f'   ROC-AUC   : {roc_auc_score(y_test, y_prob_final):.4f}')
print(f'   Recall    : {recall_score(y_test, y_pred_final):.4f}')
print(f'   Threshold : {THRESH_OTIMO}')

2026/04/08 05:41:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:41:02 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run modelo_final_producao at: http://localhost:5000/#/experiments/1/runs/babb9a2d385f4019a30fcf245e24edf7
🧪 View experiment at: http://localhost:5000/#/experiments/1
✅ Modelo final registrado no MLflow!
   Run ID    : babb9a2d385f4019a30fcf245e24edf7
   ROC-AUC   : 0.8192
   Recall    : 0.8168
   Threshold : 0.473


## Comparação Final no MLflow

In [ ]:
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(EXPERIMENTO)

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=['metrics.roc_auc DESC']
)

tipos_validos = ['baseline', 'comparacao_inicial', 'modelo_final']

linhas = []
for r in runs:
    tipo = r.data.tags.get('tipo', '')

    if tipo in tipos_validos and 'roc_auc' in r.data.metrics:
        linhas.append({
            'Run'      : r.data.tags.get('mlflow.runName', ''),
            'Tipo'     : tipo,
            'ROC-AUC'  : r.data.metrics.get('roc_auc'),
            'Recall'   : r.data.metrics.get('recall',
                         r.data.metrics.get('recall_thresh_otimo')),
            'Precisão' : r.data.metrics.get('precisao',
                         r.data.metrics.get('precisao_thresh_otimo')),
            'F1'       : r.data.metrics.get('f1',
                         r.data.metrics.get('f1_thresh_otimo')),
        })

df_comp = pd.DataFrame(linhas).sort_values('ROC-AUC', ascending=False)

print('📊 Comparação de todas as runs registradas:')
print(df_comp.to_string(index=False))

📊 Comparação de todas as runs registradas:
                         Run               Tipo  ROC-AUC   Recall  Precisão       F1
comparacao_gradient_boosting comparacao_inicial 0.819464 0.166975  0.589834 0.260271
       modelo_final_producao       modelo_final 0.819195 0.816783  0.305891 0.445091
       modelo_final_producao       modelo_final 0.819195 0.816783  0.305891 0.445091
         comparacao_lightgbm comparacao_inicial 0.818994 0.784300  0.320468 0.455015
          comparacao_xgboost comparacao_inicial 0.818795 0.784727  0.322426 0.457058
    comparacao_random_forest comparacao_inicial 0.816340 0.773330  0.318807 0.451487
baseline_logistic_regression           baseline 0.812479 0.766918  0.317731 0.449313
    comparacao_decision_tree comparacao_inicial 0.808032 0.778886  0.306137 0.439522
